# Fase 3 — Calidad de datos y limpieza

Proyecto: Detección temprana de pérdida de paridad en stablecoins (Binance)

La limpieza se ejecuta con Spark en `src/quality.py`: lee los klines horarios de la zona `raw`, aplica las reglas de calidad y escribe el resultado en la zona `processed` (Parquet particionado por símbolo/año/mes). Este notebook **no repite la limpieza**: lee ese resultado (`data/klines_clean`) y recalcula con pandas los controles de calidad, para que cualquiera pueda comprobar que las cifras del informe salen de los datos.

## Reglas de calidad que aplica `src/quality.py`

1. **Duplicados:** se descartan las filas repetidas por (`simbolo`, `open_time`).
2. **Rango válido:** `open`, `high`, `low` y `close` mayores que cero, `high` ≥ `low`, `high` ≥ `open`/`close` y `low` ≤ `open`/`close`. Las filas que no lo cumplen se rechazan.
3. **Huecos de tiempo:** se detecta toda diferencia entre dos `open_time` consecutivos de un mismo símbolo distinta de 3 600 000 ms (1 hora). **Los huecos no se rellenan ni se interpolan:** el criterio del proyecto es que un vacío de mercado es información.

## Reporte que imprime `src/quality.py` sobre los datos reales

Corrida de verificación del 5 de octubre de 2026 (escala 1x, salida en `hdfs:///datalake/benchmark/`; las zonas `raw` y `processed` solo se leyeron):

```
total_inicial: 51919
duplicados_removidos: 0
rechazados_por_rango: 0
huecos_de_tiempo_detectados: 7
total_final: 51919
```

Para reproducir la limpieza completa:

```
spark-submit --deploy-mode client --master yarn src/quality.py \
    --input hdfs:///datalake/raw/binance/klines \
    --output hdfs:///datalake/processed/binance/klines_clean
```

## 1. Carga de los datos limpios

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RUTA_DATOS = Path("../data/klines_clean")
INTERVALO_MS = 60 * 60 * 1000   # un kline horario

df = pd.read_parquet(RUTA_DATOS)
df["simbolo"] = df["simbolo"].astype(str)
df["anio"] = df["anio"].astype(int)
df["mes"] = df["mes"].astype(int)
df = df.sort_values(["simbolo", "open_time"]).reset_index(drop=True)

print(f"Total de filas: {len(df):,}")
print(f"Símbolos: {sorted(df['simbolo'].unique())}")

Total de filas: 51,919
Símbolos: ['BTCBUSD', 'BTCUSDC', 'BTCUSDT', 'BUSDUSDT', 'USDCUSDT']


## 2. Controles de calidad recalculados

Se repiten sobre los datos limpios los mismos controles de `src/quality.py`. Como el resultado ya está limpio, se espera **0 duplicados** y **0 filas con rango inválido**; los huecos de tiempo sí persisten porque no se rellenan.

In [2]:
duplicados = int(df.duplicated(["simbolo", "open_time"]).sum())

rango_valido = (
    (df["open"] > 0) & (df["high"] > 0) & (df["low"] > 0) & (df["close"] > 0)
    & (df["high"] >= df["low"])
    & (df["high"] >= df["open"]) & (df["high"] >= df["close"])
    & (df["low"] <= df["open"]) & (df["low"] <= df["close"])
)
fuera_de_rango = int((~rango_valido).sum())

dif_ms = df.groupby("simbolo")["open_time"].diff()
huecos_n = int((dif_ms.notna() & (dif_ms != INTERVALO_MS)).sum())

pd.DataFrame({
    "control": ["filas totales", "duplicados (simbolo, open_time)",
                "filas con rango inválido", "huecos de tiempo"],
    "resultado": [f"{len(df):,}", duplicados, fuera_de_rango, huecos_n],
})

,control,resultado
0,filas totales,"51,919"
1,"duplicados (simbolo, open_time)",0
2,filas con rango inválido,0
3,huecos de tiempo,7


## 3. Filas y cobertura por símbolo

Las horas de este notebook están en UTC (se calculan desde `open_time`, que es milisegundos desde 1970 en UTC).

In [3]:
cobertura = df.groupby("simbolo").agg(
    filas=("open_time", "size"),
    primer_dato_utc=("open_time", lambda s: pd.to_datetime(s.min(), unit="ms", utc=True).strftime("%Y-%m-%d %H:%M")),
    ultimo_dato_utc=("open_time", lambda s: pd.to_datetime(s.max(), unit="ms", utc=True).strftime("%Y-%m-%d %H:%M")),
)
cobertura

,filas,primer_dato_utc,ultimo_dato_utc
simbolo,,,
BTCBUSD,3648,2022-10-01 00:00,2023-03-10 23:00
BTCUSDC,13580,2022-01-01 00:00,2023-12-31 23:00
BTCUSDT,17519,2022-01-01 00:00,2023-12-31 23:00
BUSDUSDT,3648,2022-10-01 00:00,2023-03-10 23:00
USDCUSDT,13524,2022-01-01 00:00,2023-12-31 23:00


## 4. Filas por símbolo, año y mes

Es la tabla que se usa en el informe. Las columnas `anio` y `mes` son las de las particiones del Parquet, que `src/quality.py` calcula con la zona horaria de la sesión de Spark (hora de Lima, UTC-5); por eso las primeras horas de un mes en UTC pueden contarse en el mes anterior.

In [4]:
pd.set_option("display.max_rows", 100)
df.pivot_table(index=["anio", "mes"], columns="simbolo", values="open_time",
               aggfunc="count", fill_value=0)

simbolo   BTCBUSD  BTCUSDC  BTCUSDT  BUSDUSDT  USDCUSDT
anio mes                                               
2021 12         0        5        5         0         5
2022 1          0      744      744         0       744
     2          0      672      672         0       672
     3          0      744      744         0       744
     4          0      720      720         0       720
     5          0      744      744         0       744
     6          0      720      720         0       720
     7          0      744      744         0       744
     8          0      744      744         0       744
     9          5      670      720         5       598
     10       744        0      744       744         0
     11       720        0      720       720         0
     12       744        0      744       744         0
2023 1        744        0      744       744         0
     2        667        0      672       667         0
     3         24      478      743        24       494
     4          0      720      720         0       720
     5          0      744      744         0       744
     6          0      720      720         0       720
     7          0      744      744         0       744
     8          0      744      744         0       744
     9          0      720      720         0       720
     10         0      744      744         0       744
     11         0      720      720         0       720
     12         0      739      739         0       739

## 5. Huecos de tiempo detectados (no se rellenan)

Para cada hueco se muestra el último dato antes y el primer dato después, en UTC. `horas_faltantes` es la duración del hueco menos la hora que sí hay.

In [5]:
df["open_time_anterior"] = df.groupby("simbolo")["open_time"].shift()
h = df[df["open_time_anterior"].notna()
       & ((df["open_time"] - df["open_time_anterior"]) != INTERVALO_MS)].copy()

h["gap_horas"] = ((h["open_time"] - h["open_time_anterior"]) / 3_600_000).round().astype(int)
h["horas_faltantes"] = h["gap_horas"] - 1
fmt = "%Y-%m-%d %H:%M"
h["ultimo_dato_antes_utc"] = pd.to_datetime(h["open_time_anterior"], unit="ms", utc=True).dt.strftime(fmt)
h["primer_dato_despues_utc"] = pd.to_datetime(h["open_time"], unit="ms", utc=True).dt.strftime(fmt)

COLS = ["ultimo_dato_antes_utc", "primer_dato_despues_utc", "gap_horas", "horas_faltantes"]
tabla_huecos = (h[["simbolo"] + COLS]
                .sort_values(["primer_dato_despues_utc", "simbolo"])
                .reset_index(drop=True))
print(f"Huecos detectados: {len(tabla_huecos)}")
tabla_huecos

Huecos detectados: 7


,simbolo,ultimo_dato_antes_utc,primer_dato_despues_utc,gap_horas,horas_faltantes
0,BTCBUSD,2023-02-28 23:00,2023-03-10 00:00,217,216
1,BUSDUSDT,2023-02-28 23:00,2023-03-10 00:00,217,216
2,USDCUSDT,2022-09-26 02:00,2023-03-11 14:00,3996,3995
3,BTCUSDC,2022-09-29 02:00,2023-03-12 06:00,3940,3939
4,BTCUSDC,2023-03-24 12:00,2023-03-24 14:00,2,1
5,BTCUSDT,2023-03-24 12:00,2023-03-24 14:00,2,1
6,USDCUSDT,2023-03-24 12:00,2023-03-24 14:00,2,1


### El hueco de BTCBUSD es el mismo que el de BUSDUSDT

BTCBUSD es el proxy de BTCUSDC y BUSDUSDT el proxy de USDCUSDT (ver `docs/decisiones_abiertas.md`). Los dos pares BUSD tienen datos horarios de octubre de 2022 a febrero de 2023 y, tras un hueco de 9 días, solo el 10 de marzo de 2023. Es el límite de cobertura de BUSD en el origen, y es el mismo en ambos pares.

In [6]:
a = tabla_huecos[tabla_huecos["simbolo"] == "BTCBUSD"][COLS].reset_index(drop=True)
b = tabla_huecos[tabla_huecos["simbolo"] == "BUSDUSDT"][COLS].reset_index(drop=True)
print("Hueco de BTCBUSD idéntico al de BUSDUSDT:", a.equals(b))
a

Hueco de BTCBUSD idéntico al de BUSDUSDT: True


,ultimo_dato_antes_utc,primer_dato_despues_utc,gap_horas,horas_faltantes
0,2023-02-28 23:00,2023-03-10 00:00,217,216


## 6. Figura de huecos

La figura `informe/fig_huecos_tiempo.png` muestra estos huecos sobre la línea de tiempo, con un panel por símbolo (cinco en total).

![Huecos de tiempo por símbolo](../informe/fig_huecos_tiempo.png)